# LangGraph 201：构建电子邮件代理

在此笔记本中，我们将逐步在 LangGraph 中设置 **电子邮件代理**。我们将从一个简单的 ReAct 风格的代理开始，并向工作流程中添加额外的步骤，模拟真实的电子邮件助理。

 <img src="../../images/email_agent.png" style="width: auto; max-height: 500px; border-radius: 8px;"> 

要更深入地了解 LangGraph 原语并学习我们的框架，请查看我们的 [LangChain 学院]( https://academy.langchain.com/courses/intro-to-langgraph)!

## 准备工作：设置

### 加载环境变量并设置我们的模型

首先，让我们从 .env 文件加载环境变量。确保包含 .env.example 中所需的所有密钥！
我们在此示例中使用 OpenAI，但您可以随意将 ChatOpenAI 与您喜欢的其他模型提供商交换。

In [ ]:
# 将项目根添加到 Python 路径，以便我们可以从 utils 模块导入
import sys 
from pathlib import Path 
project_root =Path ().resolve ().parent .parent 
if str (project_root )not in sys .path :
    sys .path .insert (0 ,str (project_root ))

    # 从集中式utils模块导入模型
    # 这避免了笔记本之间的代码重复并使用一致的模型配置
from utils .models import model 

# 替代方案：如果您想内联定义模型而不是使用集中式配置，请取消注释以下内容：
# 从 dotenv 导入 load_dotenv
# load_dotenv(dotenv_path="../.env", override=True)
# 从 langchain.chat_models 导入 init_chat_model
# 模型 = init_chat_model("openai:o3-mini")

# 注意：对于其他提供商（Azure、Bedrock、Vertex AI），请更新 utils/models.py
# 有关切换 LLM 提供商的详细说明，请参阅 utils/models.py

### 设置短期和长期记忆

我们还将初始化一个用于**短期内存**的检查指针，在单个线程中维护上下文。 

**长期记忆**可让您存储和回忆对话之间的信息。今天，我们将利用长期记忆存储来存储用户偏好以实现个性化。

In [ ]:
from langgraph .checkpoint .memory import MemorySaver 
from langgraph .store .memory import InMemoryStore 

# 初始化长期记忆存储
in_memory_store =InMemoryStore ()

# 初始化线程级内存的检查点
checkpointer =MemorySaver ()

## 第 1 部分：创建电子邮件代理

 <img src="../../images/email_response_agent.png" style="width: auto; max-height: 500px; border-radius: 8px;">

### 状态

信息如何通过这些步骤流动？  

状态是我们将介绍的第一个 LangGraph 概念。 **状态可以被认为是代理的内存 - 它是在图的节点之间传递的共享数据结构**，代表应用程序的当前快照。 

为此，我们的电子邮件代理将跟踪以下元素： 
1. 输入电子邮件
2. 分类决策——是否回复邮件
3. 通话记录
4.任何加载的长期记忆
5. 剩余步骤 - 在达到递归限制之前

In [ ]:
from typing_extensions import TypedDict 
from typing import Annotated ,Literal 
from langgraph .graph .message import AnyMessage ,add_messages 

class State (TypedDict ):
# 该状态类具有内置的消息键
    email_input :dict 
    classification_decision :Literal ["ignore","respond","notify"]
    messages :Annotated [list [AnyMessage ],add_messages ]
    loaded_memory :str 
    remaining_steps :int 



### 工具

让我们定义我们的代理可以访问的**工具**列表。工具是可以作为法学硕士能力扩展的功能。在我们的例子中，我们将首先创建几个与 Chinook 音乐数据库交互的工具。 

我们可以使用@tool装饰器来创建工具

In [ ]:
from langchain_core .tools import tool 
from datetime import datetime 
from pydantic import BaseModel 

@tool 
def schedule_meeting (
attendees :list [str ],subject :str ,duration_minutes :int ,preferred_day :datetime ,start_time :int 
)->str :
    """安排日历会议。"""
    # 占位符响应 - 在实际应用程序中将检查日历和时间表
    date_str =preferred_day .strftime ("%A, %B %d, %Y")
    return f"Meeting '{subject}' scheduled on {date_str} at {start_time} for {duration_minutes} minutes with {len(attendees)} attendees"

@tool 
def check_calendar_availability (day :str )->str :
    """检查给定日期的日历可用性。"""
    # 占位符响应 - 在实际应用程序中将检查实际日历
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"


@tool 
def write_email (to :str ,subject :str ,content :str )->str :
    """编写并发送电子邮件。"""
    # 占位符响应 - 在实际应用程序中会发送电子邮件
    return f"邮件已发送给 {to}，主题为“{subject}”，内容为：{content}"

@tool 
class Done (BaseModel ):
    """电子邮件已发送。"""
    done :bool 

为了让我们的法学硕士知道这些工具可以调用，我们将使用 `bind_tools` 方法。

In [ ]:
tools = [schedule_meeting, check_calendar_availability, write_email, Done]
tools_by_name = {tool.name: tool for tool in tools}

llm_with_tools = model.bind_tools(tools, tool_choice="any")

### 节点

现在我们有了工具列表，我们准备构建与它们交互的节点。 

节点只是 python（或 JS/TS！）函数。节点将图形的状态作为输入，执行一些逻辑，然后返回一个新的状态。 

在这里，我们将为 ReAct 代理设置 2 个节点：
1. **reasoning**：推理节点，决定调用哪个函数 
2. **tools**：包含所有可用工具并执行功能的节点

LangChain 有一个 ToolNode，我们可以利用它为我们的工具创建一个节点。

In [ ]:
from langgraph.prebuilt import ToolNode
# Node
tool_node = ToolNode(tools)

对于电子邮件处理，让我们定义帮助器来解析和格式化电子邮件输入。

In [ ]:
def parse_email(email_input: dict) -> dict:
    return (
        email_input["author"],
        email_input["to"],
        email_input["subject"],
        email_input["email_thread"],
    )

def format_email_markdown(subject, author, to, email_thread, email_id=None):
    id_section = f"\n**ID**: {email_id}" if email_id else ""
    
    return f"""

**Subject**: {subject}
**From**: {author}
**To**: {to}{id_section}

{email_thread}

---
"""


In [ ]:
from langchain .messages import SystemMessage ,HumanMessage 

def create_agent_prompt (state :State ):
    action_instructions ="""<角色>
    您是一位一流的行政助理，关心帮助您的主管尽可能地表现出色。
    </角色>

<工具>
    您可以使用以下工具来帮助管理通信和日程安排：

1. write_email(to, subject, content) - 发送电子邮件给指定收件人
    2.schedule_meeting(attendees,subject,duration_months,preferred_day,start_time) - 安排日历会议
    3. check_calendar_availability(day) - 检查给定日期的可用时段
    4. 完成 - 电子邮件已发送
    </工具>

< 使用说明 >
    处理电子邮件时，请按以下步骤操作：
    1.仔细分析邮件内容和目的
    3. 要回复电子邮件，请使用 write_email 工具起草回复电子邮件
    4. 对于会议请求，请使用 check_calendar_availability 工具查找空闲时间段
    5. 要安排会议，请使用schedule_meeting 工具，并为preferred_day 参数使用日期时间对象
    - 今天的日期是 {today} - 使用它来准确安排会议
    6. 如果您安排了会议，请使用 write_email 工具起草一封简短的回复电子邮件
    7、使用write_email工具后，任务完成
    8. 如果您已发送电子邮件，则使用“完成”工具指示任务已完成
    </说明>

<背景>
    我是 Robert，浪链的软件工程师。
    </背景>

< 响应偏好 >
    使用专业、简洁的语言。如果电子邮件提到截止日期，请务必在回复中明确确认并提及截止日期。

在回答需要调查的技术问题时：
    - 明确说明您是否会调查或询问谁
    - 提供预计的时间表，以便您了解更多信息或完成任务

回复活动或会议邀请时：
    - 务必确认任何提到的截止日期（特别是注册截止日期）
    - 如果提到研讨会或特定主题，请询问有关它们的更多具体细节
    - 如果提到折扣（团体或早鸟），请明确要求提供相关信息
    - 不要承诺

在响应协作或项目相关请求时：
    - 确认提及的任何现有工作或材料（草稿、幻灯片、文档等）
    - 明确提及在会议之前或会议期间审查这些材料
    - 安排会议时，明确说明建议的具体日期、日期和时间。

响应会议安排请求时：
    - 如果收件人要求会议承诺，请验证原始电子邮件中提到的所有时间段的可用性，然后根据您的可用性通过安排会议来承诺建议的时间之一。或者，说您无法在建议的时间到达。
    - 如果询问是否有空位，请检查您的日历是否有空位，并在可用时发送一封电子邮件，提出多个时间选项。不要安排会议
    - 在您的回复中提及会议持续时间，以确认您已正确记录。
    - 在您的回复中提及会议的目的。
    </响应偏好>

<日历首选项>
    30 分钟的会议是首选，但 15 分钟的会议也可以接受。
    当天晚些时候的时间是最好的。
    </日历首选项>"""

    email =parse_email (state ["email_input"])
    email_markdown =format_email_markdown (email [2 ],email [0 ],email [1 ],email [3 ])
    email_request =f"请回复以下邮件： {email_markdown}"

    prompt =[
    SystemMessage (action_instructions .format (today =datetime .now ().strftime ("%Y-%m-%d"))),
    HumanMessage (content =email_request )
    ]

    return prompt +state ["messages"]

    # Nodes
def reasoning_node (state :State ):
    """LLM决定是否调用工具"""
    prompt =create_agent_prompt (state )
    result =llm_with_tools .invoke (prompt )

    return {"messages":[result ]}

### 边缘

现在，我们需要定义一个连接我们定义的节点之间的控制流，这就是边概念的用武之地。

**边是节点之间的连接。它们定义了图表的流程。**
* **法线**是确定性的，并且总是从一个节点到其定义的目标
* **条件边** 用于在节点之间动态路由，实现为基于某种逻辑返回下一个要访问的节点的函数。 

在这种情况下，我们需要来自子代理的**条件边缘**来确定是否： 
- 调用工具，或者，
- 如果用户查询完成则路由到末尾

In [ ]:
from langgraph .graph import END ,START 

def should_continue (state :State )->Literal ["Tools","__end__"]:
    """路由至工具，或者如果调用完成工具则结束"""
    messages =state ["messages"]
    last_message =messages [-1 ]
    if last_message .tool_calls :
        for tool_call in last_message .tool_calls :
            if tool_call ["name"]=="Done":
                return END 
            else :
                return "Tools"

### 编译图表！

现在我们已经定义了状态和节点，让我们将它们放在一起并构建我们的反应代理！

In [ ]:
from langgraph .graph import StateGraph 

# 构建工作流程
agent_builder =StateGraph (State )

# 添加节点
agent_builder .add_node ("agent",reasoning_node )
agent_builder .add_node ("tools",tool_node )

# 添加边来连接节点
agent_builder .add_edge (START ,"agent")
agent_builder .add_conditional_edges (
"agent",
should_continue ,
{
# should_continue 返回的名称：下一个要访问的节点的名称
"Tools":"tools",
END :END ,
},
)
agent_builder .add_edge ("tools","agent")

# 编译代理
agent =agent_builder .compile (checkpointer =checkpointer ,store =in_memory_store )
agent 

### 测试
让我们看看我们的代理如何回复电子邮件！

In [ ]:
from langsmith import uuid7 

config ={"configurable":{"thread_id":uuid7 ()}}

email_input ={
"to":'徐罗伯特 <Robert@company.com>',
"author":'团队负责人 <teamlead@company.com>',
"subject":'季度计划会议',
"email_thread":'嗨罗伯特，\n\n现在是我们季度计划会议的时间了。我想在下周安排一次 90 分钟的会议来讨论我们的第三季度路线图。\n\n您能告诉我您周一或周三有空吗？最好是上午 10 点到下午 3 点之间的某个时间。\n\n期待您对新功能优先级的意见。\n\n最好的，\n团队负责人'
}

result =agent .invoke ({"email_input":email_input },config =config )

print ('回复电子邮件：')
print (format_email_markdown (email_input ["subject"],email_input ["author"],email_input ["to"],email_input ["email_thread"]))

for message in result ["messages"]:
    message .pretty_print ()

## 第 2 部分：使用预构建

我们为电子邮件代理创建的架构是一种称为 ReAct 代理的通用架构。这种工具调用格式非常流行，因此LangChain实际上提供了一个预构建的函数来轻松启动ReAct代理。

让我们使用预构建的代理重新实现我们的代理！

In [ ]:
from langchain .agents import create_agent 
from datetime import datetime 

# 为create_agent创建静态系统提示字符串
# 电子邮件上下文将通过状态中的用户消息提供
action_instructions ="""<角色>
您是一位一流的行政助理，关心帮助您的主管尽可能地表现出色。
</角色>

<工具>
您可以使用以下工具来帮助管理通信和日程安排：

1. write_email(to, subject, content) - 发送电子邮件给指定收件人
2.schedule_meeting(attendees,subject,duration_months,preferred_day,start_time) - 安排日历会议
3. check_calendar_availability(day) - 检查给定日期的可用时段
4. 完成 - 电子邮件已发送
</工具>

< 使用说明 >
处理电子邮件时，请按以下步骤操作：
1.仔细分析邮件内容和目的
3. 要回复电子邮件，请使用 write_email 工具起草回复电子邮件
4. 对于会议请求，请使用 check_calendar_availability 工具查找空闲时间段
5. 要安排会议，请使用schedule_meeting 工具，并为preferred_day 参数使用日期时间对象
- 今天的日期是 {today} - 使用它来准确安排会议
6. 如果您安排了会议，请使用 write_email 工具起草一封简短的回复电子邮件
7、使用write_email工具后，任务完成
8. 如果您已发送电子邮件，则使用“完成”工具指示任务已完成
</说明>

<背景>
我是 Robert，浪链的软件工程师。
</背景>

< 响应偏好 >
- 保持回复专业但友好
- 简明扼要
- 如果安排会议，请尽可能提出 2-3 个时间选项
</响应偏好>"""

# 使用今天的日期设置格式
system_prompt_string =action_instructions .format (today =datetime .now ().strftime ("%Y-%m-%d"))

# 定义子代理
email_prebuilt =create_agent (
model =model ,
tools =tools ,
name ="email_prebuilt",
system_prompt =system_prompt_string ,
state_schema =State ,
checkpointer =checkpointer ,
store =in_memory_store 
)
email_prebuilt 

让我们继续测试我们的预构建！

In [ ]:
from langsmith import uuid7 

config ={"configurable":{"thread_id":uuid7 ()}}

email_input ={
"to":'徐罗伯特 <Robert@company.com>',
"author":'团队负责人 <teamlead@company.com>',
"subject":'季度计划会议',
"email_thread":'嗨罗伯特，\n\n现在是我们季度计划会议的时间了。我想在下周安排一次 90 分钟的会议来讨论我们的第三季度路线图。\n\n您能告诉我您周一或周三有空吗？最好是上午 10 点到下午 3 点之间的某个时间。\n\n期待您对新功能优先级的意见。\n\n最好的，\n团队负责人'
}

from langchain_core .messages import HumanMessage 

result =email_prebuilt .invoke ({
"email_input":email_input ,
"messages":[
HumanMessage (content =f"""请回复以下邮件：

**Subject**: {email_input['subject']}
**From**: {email_input['author']}
**To**: {email_input['to']}

{email_input['email_thread']}
""")
]
},config =config )

print ('回复电子邮件：')
print (format_email_markdown (email_input ["subject"],email_input ["author"],email_input ["to"],email_input ["email_thread"]))

for message in result ["messages"]:
    message .pretty_print ()

## 第 3 部分：人机交互

 <img src="../../images/email_agent.png" style="width: auto; max-height: 500px; border-radius: 8px;"> 

在许多代理中，高可靠性代理在采取行动之前需要人工批准或反馈。在我们的例子中，我们的电子邮件代理应该确定是否忽略或回复电子邮件，但它可能并不总是能够自行做出该决定。

因此，我们将允许电子邮件代理在处理电子邮件时向人类寻求帮助。我们还将演示如何将此功能引入涉及多个代理步骤的更复杂的工作流程中。

我们将利用**人机交互**为法学硕士提供关于是否忽略或回复电子邮件的人工反馈。

### 添加分类功能

首先，让我们在工作流程中添加一个新步骤。我们希望代理能够对电子邮件进行分类并决定它们是否值得回复。为此，我们将添加一个新的“分类”节点。

我们将首先为我们的法学硕士定义一个模式，以遵循其输出。

In [ ]:
from pydantic import BaseModel ,Field 

class RouterSchema (BaseModel ):
    """分析未读电子邮件并根据其内容进行路由。"""

    reasoning :str =Field (
    description ='分类背后的逐步推理。'
    )
    classification :Literal ["ignore","respond","notify"]=Field (
    description ='电子邮件的分类：“忽略”不相关的电子邮件，'
    '“通知”不需要回复的重要信息，'
    '“回复”需要回复的电子邮件',
    )

llm_router =model .with_structured_output (RouterSchema )

然后，我们将创建分类节点本身及其提示。

In [ ]:
from langchain .messages import SystemMessage ,HumanMessage 

def create_triage_prompt (state :State ):
    loaded_memory =""
    if "loaded_memory"in state :
        loaded_memory =state ["loaded_memory"]

    triage_instructions =f"""
    < Role >
    你的角色是根据下面的说明和背景信息，对收到的邮件进行分流。
    </ Role >

    < Background >
    我是 Robert，LangChain 的软件工程师。
    </ Background >

    < Instructions >
    将每封邮件归入以下三类之一：
    1. IGNORE - 不值得回复或跟踪的邮件
    2. NOTIFY - 值得通知的重要信息，但不需要回复
    3. RESPOND - 需要直接回复的邮件
    请将下面的邮件归入其中一类。
    </ Instructions >

    < Rules >
    不值得回复的邮件包括：
    - 营销通讯和促销邮件
    - 垃圾邮件或可疑邮件
    - 只是抄送的 FYI 线程，且没有直接问题

    还有一些事情应该被知晓，但不需要邮件回复。对于这些情况，你应该通知（使用 `notify` 响应）。示例包括：
    - 团队成员病假或休假
    - 构建系统通知或部署通知
    - 没有行动项的项目状态更新
    - 重要公司公告
    - 包含当前项目相关信息的 FYI 邮件
    - HR 部门的截止日期提醒
    - GitHub 通知

    值得回复的邮件包括：
    - 团队成员提出、需要专业知识回答的直接问题
    - 需要确认的会议请求
    - 与团队项目相关的关键 bug 报告
    - 管理层提出、需要确认收到的请求
    - 客户关于项目状态或功能的询问
    - 关于文档、代码或 API 的技术问题（尤其是缺失端点或功能的问题）
    - 与家庭相关的个人提醒（妻子 / 女儿）
    - 与自我照护相关的个人提醒（医生预约等）
    </ Rules >

    {loaded_memory}
    """
    author ,to ,subject ,email_thread =parse_email (state ["email_input"])
    email_markdown =format_email_markdown (subject ,author ,to ,email_thread )
    email_request =f"请判断如何处理下面的邮件线程：{email_markdown}"
    prompt =[
    SystemMessage (triage_instructions ),
    HumanMessage (content =email_request )
    ]
    # 仅在消息存在且不为空时追加消息
    if "messages"in state and state ["messages"]:
        prompt =prompt +state ["messages"]
    return prompt 

def triage_router (state :State ):
    """分析电子邮件内容来决定我们是否应该回复、通知或忽略。"""
    prompt =create_triage_prompt (state )
    # 运行路由器 LLM - 使用已配置结构化输出的 llm_router
    result =llm_router .invoke (prompt )

    # Decision
    classification =result .classification 
    return {"classification_decision":classification }


当代理决定 `notify` 人类时，我们还将创建一个节点来添加人类反馈。这些是代理归类为重要的电子邮件，但不值得回复。我们将让人类用户验证这些重要的电子邮件是否确实值得回复。

In [ ]:
from langgraph .types import interrupt 

# Node
def human_input (state :State ):
    """纳入人类反馈的节点"""
    author ,to ,subject ,email_thread =parse_email (state ["email_input"])
    email_markdown =format_email_markdown (subject ,author ,to ,email_thread )
    user_input =interrupt (f"请判断下面这封邮件是否值得回复 (Y/n)：{email_markdown}")

    log ='电子邮件最初标记为通知，但用户将其标记为需要响应的场景。'
    if str (user_input ).lower ()=="y":
        return {"classification_decision":"respond","messages":[HumanMessage (content =log )]}
    else :
        return {"classification_decision":"ignore"}


最后，我们将定义我们需要的新边。如果分类步骤返回 `notify` 分类，我们将定义一条触发“Human In the Loop”的边缘，以及从人类反馈步骤到代理其余部分的边缘。

In [ ]:
def handle_classification (state :State ):
    """如果电子邮件被分类为通知，则触发人工审核"""
    if state ["classification_decision"]=="notify":
        return "human_input"
    elif state ["classification_decision"]=="respond":
        return "email_agent"
    else :
        return END 


def handle_human_input (state :State ):
    """处理人工输入"""
    if state ["classification_decision"]=="respond":
        return "email_agent"
    else :
        return END 


让我们来编译我们的代理吧！

In [ ]:
email_hitl_workflow = StateGraph(State)
email_hitl_workflow.add_node("triage", triage_router)
email_hitl_workflow.add_node("human_input", human_input)
email_hitl_workflow.add_node("email_agent", agent)
email_hitl_workflow.add_edge(START, "triage")
email_hitl_workflow.add_conditional_edges(
    "triage",
    handle_classification,
    {
        "human_input": "human_input",
        "email_agent": "email_agent",
        END: END,
    },
)
email_hitl_workflow.add_conditional_edges(
    "human_input",
    handle_human_input,
    {
        "email_agent": "email_agent",
        END: END,
    },
)
email_hitl = email_hitl_workflow.compile(checkpointer=checkpointer, store=in_memory_store)
email_hitl



### 测试

让我们调用我们的新代理！

In [ ]:
from langsmith import uuid7 

config ={"configurable":{"thread_id":uuid7 ()}}

email_input ={
"to":'徐罗伯特 <Robert@company.com>',
"author":'系统管理员 <sysadmin@company.com>',
"subject":'定期维护-数据库停机',
"email_thread":'嗨罗伯特，\n\n谨此提醒您，我们将于今晚美国东部时间凌晨 2 点至 4 点对生产数据库执行计划维护。在此期间，所有数据库服务将不可用。\n\n请相应地规划您的工作，并确保在此窗口期间不会安排任何关键部署。\n\n谢谢，\n系统管理团队'
}

result =email_hitl .invoke ({"email_input":email_input },config =config )

if "__interrupt__"in result :
    print ('分类决策：',result ["classification_decision"])
    print ('----- 需要人工协助 -----')
    print (result ["__interrupt__"][0 ].value )
else :
    for message in result ["messages"]:
        message .pretty_print ()


In [ ]:
from langgraph.types import Command

result = email_hitl.invoke(Command(resume="y"), config=config)

for message in result["messages"]:
    message.pretty_print()

## [可选]第 4 部分：内存

现在我们已经创建了一个包含验证和执行的代理工作流程，让我们更进一步。 

**长期记忆**可让您存储和回忆对话之间的信息。我们已经初始化了一个长期记忆存储。 

 <img src="../../images/email_agent_memory.png" style="width: auto; max-height: 500px; border-radius: 8px;"> 

在此步骤中，我们将添加 2 个节点： 
- **load_memory** 从长期内存存储加载的节点
- **create_memory** 节点，保存客户分享的关于他们自己的任何音乐兴趣

让我们从加载内存开始。这将使我们能够个性化我们希望代理回复的电子邮件，而无需将其硬编码到提示中。

In [ ]:
from langgraph .store .base import BaseStore 

# 构造内存的辅助函数
def format_user_memory (user_data ):
    """格式化用户的音乐偏好（如果有）。"""
    profile =user_data ['memory']
    result ='<附加规则>'
    result +='以下是用户认为重要的自定义规则。请优先考虑这些规则：'
    if hasattr (profile ,'response_preferences')and profile .response_preferences :
        result +="\n- ".join (profile .response_preferences )
    result +='</附加规则>'

    return result .strip ()

    # Node
def load_memory (state :State ,store :BaseStore ):
    """加载用户的音乐偏好（如果有）。"""

    namespace =("memory_profile","Robert")
    existing_memory =store .get (namespace ,"user_memory")
    formatted_memory =""
    if existing_memory and existing_memory .value :
        formatted_memory =format_user_memory (existing_memory .value )

    return {"loaded_memory":formatted_memory }

接下来，我们将创建 create_memory 节点。我们将使用结构化输出来确保所有记忆都采用相同的格式。

In [ ]:
# 用于创建内存的用户配置文件结构
from typing import List 

class UserProfile (BaseModel ):
    response_preferences :List [str ]=Field (
    description ='描述用户想要回复什么类型的电子邮件的规则列表'
    )

In [ ]:
create_memory_prompt ="""您是一位专家分析师，正在观察客户和行政助理之间发生的对话。行政助理帮助客户处理电子邮件。
您的任务是分析客户和行政助理之间发生的交互，并更新与客户相关的内存配置文件。 
您特别关心保存客户分享的关于他们自己的任何音乐兴趣，特别是他们的音乐偏好到他们的记忆配置文件。

<核心指令>
1. 内存配置文件可能为空。如果它是空的，您应该始终为客户创建一个新的内存配置文件。
2. 您应该确定电子邮件的哪些特征导致用户想要回复它。
3. 对于内存配置文件中的每个键，如果没有新信息，则不要更新该值 - 保持现有值不变。
4. 仅当有新信息时才更新内存配置文件中的值。
</核心指令>

<预期格式>
客户的内存配置文件应具有以下字段：
-response_preferences：描述用户想要回复什么类型的电子邮件的规则列表

重要提示：确保您的响应是具有这些字段的对象。
</预期格式>


<重要上下文>
**以下重要背景**
为了帮助您完成此任务，我在下面附上了客户和客户支持助理之间进行的对话，以及您应该更新或创建的与客户关联的现有内存配置文件。 

您应该分析的客户和客户支持助理之间的对话如下：
 {conversation} 

您应该根据对话更新或创建与客户关联的现有内存配置文件，如下所示：
 {memory_profile} 

</important_context>

提醒：在做出回应之前深呼吸并仔细思考。"""

# Node
def create_memory (state :State ,store :BaseStore ):
    namespace =("memory_profile","Robert")
    formatted_memory =state ["loaded_memory"]

    email_input =state ["email_input"]
    formatted_email =format_email_markdown (email_input ["subject"],email_input ["author"],email_input ["to"],email_input ["email_thread"])
    initial_message =f"Initial email received: {formatted_email}\n"
    conversation =HumanMessage (content =initial_message )+state ["messages"]

    formatted_system_message =SystemMessage (content =create_memory_prompt .format (conversation =conversation ,memory_profile =formatted_memory ))
    # 不同的提供商需要至少一条人工消息
    updated_memory =model .with_structured_output (UserProfile ).invoke ([
    formatted_system_message ,
    HumanMessage (content ='请分析对话并更新记忆配置文件。')
    ])
    key ="user_memory"
    store .put (namespace ,key ,{"memory":updated_memory })

让我们将这些节点添加到我们的图中。仅当用户提供反馈时，我们才会创建新的记忆。换句话说，只有当用户决定回复最初标记为“仅通知”的电子邮件时，我们才会创建新的内存。

In [ ]:
def should_create_memory (state :State ):
    """仅当用户决定回复电子邮件时才创建新的内存"""
    messages =state ["messages"]

    correction ='电子邮件最初标记为通知，但用户将其标记为需要响应的场景。'
    for message in messages :
        if message .type =="human"and correction in message .content :
            return "create_memory"

    return END 


In [ ]:
email_memory_workflow = StateGraph(State)
email_memory_workflow.add_node("triage", triage_router)
email_memory_workflow.add_node("human_input", human_input)
email_memory_workflow.add_node("email_agent", agent)

email_memory_workflow.add_node("load_memory", load_memory)
email_memory_workflow.add_node("create_memory", create_memory)

email_memory_workflow.add_edge(START, "load_memory")
email_memory_workflow.add_edge("load_memory", "triage")


email_memory_workflow.add_conditional_edges(
    "triage",
    handle_classification,
    {
        "human_input": "human_input",
        "email_agent": "email_agent",
        END: END,
    },
)
email_memory_workflow.add_conditional_edges(
    "human_input",
    handle_human_input,
    {
        "email_agent": "email_agent",
        END: END,
    },
)
email_memory_workflow.add_conditional_edges(
    "email_agent",
    should_create_memory,
    {
        "create_memory": "create_memory",
        END: END,
    },
)

email_agent_memory = email_memory_workflow.compile(checkpointer=checkpointer, store=in_memory_store)
email_agent_memory

In [ ]:
from langsmith import uuid7 

config ={"configurable":{"thread_id":uuid7 (),"recursion_limit":100 }}

email_input ={
"to":'徐罗伯特 <Robert@company.com>',
"author":'系统管理员 <sysadmin@company.com>',
"subject":'定期维护-数据库停机',
"email_thread":'嗨罗伯特，\n\n谨此提醒您，我们将于今晚美国东部时间凌晨 2 点至 4 点对生产数据库执行计划维护。在此期间，所有数据库服务将不可用。\n\n请相应地规划您的工作，并确保在此窗口期间不会安排任何关键部署。\n\n谢谢，\n系统管理团队'
}

result =email_agent_memory .invoke ({"email_input":email_input },config =config )
result =email_agent_memory .invoke (Command (resume ="y"),config =config )


print ('回复电子邮件：')
print (format_email_markdown (email_input ["subject"],email_input ["author"],email_input ["to"],email_input ["email_thread"]))

for message in result ["messages"]:
    message .pretty_print ()


让我们看看我们的代理在内存中存储了什么

In [ ]:
user = "Robert"
namespace = ("memory_profile", user)
memory = in_memory_store.get(namespace, "user_memory").value

saved_preferences = memory.get("memory").response_preferences

print(saved_preferences)